In [1]:
import os
import numpy as np
import pandas as pd
import random
import pickle
import torch
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

from lime.lime_tabular import LimeTabularExplainer

In [2]:
# Set device and seed
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 316
random.seed(seed)
np.random.seed(seed)

# Load Data

In [3]:
with open('../data/train_sequences_idx.pkl', 'rb') as f:
    train_sequences_idx = pickle.load(f)
with open('../data/test_sequences_idx.pkl', 'rb') as f:
    test_sequences_idx = pickle.load(f)

with open('../data/user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)
    
with open('../data/ratings_dict.pkl', 'rb') as f:
    ratings_dict = pickle.load(f)
with open('../data/movie_index_to_title.pkl', 'rb') as f:
    movie_idx_to_title = pickle.load(f)

user_embeddings = np.load('../data/user_embeddings.npy')
movie_embeddings = np.load('../data/movie_embeddings.npy')

# Mapping from index to movie ID (if available)
with open('../data/idx2movie.pkl', 'rb') as f:
    idx2movie = pickle.load(f)

In [4]:
# Logd Model
version_number = 3
total_timesteps = 500_000
model = PPO.load(f"../models/PPO_Ver_{version_number}_{total_timesteps}")

/home/stu5/s5/law3082/miniconda3/envs/idai610/lib/python3.10/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


# Env

In [5]:
class DRRRecommendationEnv(gym.Env):
    def __init__(self, 
                 train_sequences_idx,      # dict: user_idx -> [movie_idx, ...]
                 test_sequences_idx,       # dict: user_idx -> [movie_idx, ...]
                 ratings_dict,             # (user_idx, movie_idx) -> rating
                 user_embeddings, movie_embeddings,
                 evaluable_users,
                 n_history=5, T_max=10):
        super().__init__()
        self.train = train_sequences_idx
        self.test = test_sequences_idx
        self.ratings = ratings_dict
        self.user_embeddings = user_embeddings
        self.movie_embeddings = movie_embeddings
        self.evaluable_users = evaluable_users
        self.n_history = n_history
        self.T_max = T_max

        self.k = movie_embeddings.shape[1]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(3*self.k,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.k,), dtype=np.float32)
        self.item_weights = np.ones(self.n_history)

        self.current_user = None
        self.history = None
        self.candidate_items = None
        self.recommended_set = None
        self.step_count = 0

    def _get_state(self, user_idx, history_indices):
        u = self.user_embeddings[user_idx]
        weights = np.exp(self.item_weights) / np.sum(np.exp(self.item_weights))
        hist_embs = self.movie_embeddings[history_indices]
        weighted_avg = np.sum(hist_embs * weights[:, np.newaxis], axis=0)
        u_g = u * weighted_avg
        return np.concatenate([u, u_g, weighted_avg]).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)

        # During evaluation, I'll always call reset(options={'user_idx': user_idx})
        # options lets me pass a specific user Index
        if options is not None and 'user_idx' in options:
            user_idx = options['user_idx']
        else:
            # NOTE: This should never happen b/c reset() has options passed in with user_idx
            # And the eval loop is iterating thru all user indices.
            user_idx = np.random.choice(self.evaluable_users)
        self.current_user = user_idx

        # Build positive history from training (last n_history items with rating >=4)
        pos_items = []
        for m in self.train[user_idx]:
            if self.ratings.get((user_idx, m), 0) >= 4:
                pos_items.append(m)
        self.history = pos_items[-self.n_history:].copy()

        # Set initial state
        self.candidate_items = set(self.test[user_idx])
        self.recommended_set = set()
        self.step_count = 0
        state = self._get_state(user_idx, self.history)
        return state, {}

    def step(self, action):
        candidates = list(self.candidate_items - self.recommended_set)
        if not candidates:
            next_state = self._get_state(self.current_user, self.history)
            return next_state, 0.0, True, False, {}

        cand_emb = self.movie_embeddings[candidates]
        scores = cand_emb @ action
        best_idx = np.argmax(scores)
        selected_item = candidates[best_idx]

        rating = self.ratings.get((self.current_user, selected_item), 0)
        reward = (rating - 3) / 2.0

        new_history = self.history.copy()
        if reward > 0:
            new_history = new_history[1:] + [selected_item]

        self.recommended_set.add(selected_item)
        self.step_count += 1

        terminated = (len(self.candidate_items - self.recommended_set) == 0)
        truncated = (self.step_count >= self.T_max)

        next_state = self._get_state(self.current_user, new_history)
        self.history = new_history

        info = {'selected_item': selected_item, 'rating': rating, 'reward': reward, 'step': self.step_count}
        return next_state, reward, terminated, truncated, info

In [6]:
# Create list of User Indices that you can legitimately evaluate
evaluable_users = []
for user_idx in train_sequences_idx.keys():
    # Count positive items in training
    pos_train = 0
    for movie_idx in train_sequences_idx[user_idx]:
        rating = ratings_dict.get((user_idx, movie_idx), 0)
        if rating >= 4:
            pos_train += 1

    # if # of positively rated movies in the train set < 5, skip the rest of the for loop
    if pos_train < 5:
        continue
    
    # Check test set has at least one item
    test_items = test_sequences_idx.get(user_idx, [])
    if len(test_items) == 0:
        continue
    
    # Check test set has at least one positive item
    has_positive_test = False
    for movie_idx in test_items:
        rating = ratings_dict.get((user_idx, movie_idx), 0)
        if rating >= 4:
            has_positive_test = True
            break
    
    if not has_positive_test:
        continue
    
    evaluable_users.append(user_idx)

print(f"Number of evaluable users: {len(evaluable_users)}")

Number of evaluable users: 5961


# LIME

In [ ]:
def g(active):
    return np.mean(movie_embeddings[active], axis=0)

In [7]:
# LIME wrapper for PPO
class PPOExplainerWrapper:
    def __init__(self, model, user_idx, history_indices, target_midx,
                 movie_emb, user_emb, n_history):
        self.model = model
        self.user_idx = user_idx
        self.history_indices = history_indices
        self.target_emb = movie_emb[target_midx]
        self.movie_emb = movie_emb
        self.user_emb = user_emb
        self.n_history = n_history

    def predict(self, masks):
        scores = []
        for mask in masks:
            active = [self.history_indices[i] for i in range(len(mask)) if mask[i] > 0.5]
            if not active:
                active = [self.history_indices[0]]  # fallback

            # State computation (same as environment's _get_state)
            u = self.user_emb[self.user_idx]
            # Uniform weights (softmax of ones)
            weights = np.ones(len(active)) / len(active)
            hist_embs = self.movie_emb[active]
            g = np.sum(hist_embs * weights[:, np.newaxis], axis=0)
            u_g = u * g
            state = np.concatenate([u, u_g, g]).astype(np.float32)

            action, _ = self.model.predict(state, deterministic=True)
            score = np.dot(action, self.target_emb)
            scores.append(score)
        return np.array(scores)

In [10]:
def explain_user(orig_uid, user2idx, eval_env, model, movie_idx_to_title, ratings_dict, user_embeddings, movie_embeddings, n_history=5, T_max=10):
    """
        Generate LIME explanations for a given user.
        
        Args:
        orig_uid: original user ID (must exist in user2idx)
        eval_env: evaluation environment instance (will be reset for this user)
        model: PPO model
    """
    # Choose a user (original ID, then map to index)
    user_idx = user2idx[orig_uid]
    
    # Reset environment
    obs, _ = eval_env.reset(options={'user_idx': user_idx})
    
    steps = []
    done = False
    step_count = 0
    
    while not done and step_count < T_max:
        history_before = eval_env.history.copy()   # positive history used for current state
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        steps.append({
            'step': step_count+1,
            'history': history_before,
            'rec_midx': info['selected_item'],
            'rating': info['rating']
        })
        step_count += 1
        done = terminated or truncated
    
    # Print trajectory
    print(f"Recommendations for USER {orig_uid} & their Actual Rating:")
    ground_truth_df = []
    for s in steps:
        ground_truth_df.append({
            'Step': s['step'],
            'Movie Title': movie_idx_to_title.get(s['rec_midx'], 'Unknown'),
            'Rating': s['rating']
        })
    ground_truth_df = pd.DataFrame(ground_truth_df)
    print(ground_truth_df.to_string(index=False))
    
    # For each step, explain
    for s in steps:
        print(f"\nStep {s['step']}: Recommended {movie_idx_to_title.get(s['rec_midx'], 'Unknown')} (rating {s['rating']})")
        history = s['history']
        if len(history) < 5:
            history = history + [history[0]] * (5 - len(history))
    
        history_titles = [movie_idx_to_title.get(m, 'Unknown') for m in history]
        history_ratings = [ratings_dict.get((user_idx, m), 'N/A') for m in history]
    
        # LIME explainer
        background = np.random.randint(0, 2, size=(500, 5))
        explainer = LimeTabularExplainer(
            background,
            feature_names=history_titles,
            categorical_features=list(range(5)),
            mode='regression'
        )
    
        wrapper = PPOExplainerWrapper(
            model, user_idx, history, s['rec_midx'],
            movie_embeddings, user_embeddings, n_history=5
        )
    
        exp = explainer.explain_instance(np.ones(5), wrapper.predict, num_features=5)
    
        # Format output (clean titles)
        exp_list = exp.as_list()
        title_to_rating = dict(zip(history_titles, history_ratings))
        df = pd.DataFrame(exp_list, columns=["Movie in History", "Weight"])
        df["Movie in History"] = df["Movie in History"].str.split('=').str[0].str.split('>').str[0].str.strip()
        df["Rating"] = df["Movie in History"].map(title_to_rating)
        df["Influence"] = df["Weight"].apply(lambda w: '+' if w > 0 else '-')
        df["Weight"] = df["Weight"].round(2)
        df = df.sort_values("Weight", ascending=False).head(5)
        print(df[["Movie in History", "Rating", "Weight", "Influence"]].to_string())

In [11]:
users_to_explain = [1, 3, 40, 67, 316]   # user IDs

eval_env = DRRRecommendationEnv(
    train_sequences_idx=train_sequences_idx,
    test_sequences_idx=test_sequences_idx,
    ratings_dict=ratings_dict,
    user_embeddings=user_embeddings,
    movie_embeddings=movie_embeddings,
    evaluable_users=evaluable_users,
    n_history=5,
    T_max=10
)

for uid in users_to_explain:
    explain_user(uid, user2idx, eval_env, model, movie_idx_to_title, ratings_dict,
                 user_embeddings, movie_embeddings)
    print("_"*80)

Recommendations for USER 1 & their Actual Rating:
 Step                         Movie Title  Rating
    1                    Toy Story (1995)     5.0
    2                Bug's Life, A (1998)     5.0
    3         Beauty and the Beast (1991)     5.0
    4                        Mulan (1998)     4.0
    5                      Aladdin (1992)     4.0
    6               Close Shave, A (1995)     3.0
    7                         Antz (1998)     4.0
    8                       Tarzan (1999)     3.0
    9 Hunchback of Notre Dame, The (1996)     4.0
   10                   Pocahontas (1995)     5.0

Step 1: Recommended Toy Story (1995) (rating 5.0)
                         Movie in History  Rating  Weight Influence
0                 Schindler's List (1993)     5.0    0.13         +
1  Snow White and the Seven Dwarfs (1937)     4.0    0.04         +
3           Miracle on 34th Street (1947)     4.0   -0.01         -
4                          Ponette (1996)     4.0   -0.01         -
2        